## STEP 1 : 데이터 전처리

### 모듈 다운로드

### 데이터 다운로드 from kaggle

### EDA
**Train 이미지 - Annotation**
- 수량 확인 및 비교
- 클래스 별 이미지 분포 (표 & 그래프)
- Mapping -> Visualization -> Mapping 검증

### Transform ~ Dataset ~ DataLoader

In [1]:
%pip install -q iterative-stratification

/Users/codeit/Desktop/Vault_home/💼 Projects(Work)/Sprint AI_14기/workspace/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#@title Import Module
import os
import cv2
import csv
import glob
import json
import random
import seaborn as sns
import kagglehub
import numpy as np
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torchvision

from torchvision import tv_tensors
from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection import _utils as det_utils

from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from functools import partial

from torchvision.models.detection import (
    RetinaNet_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.retinanet import (
    RetinaNetClassificationHead,
)
from torchvision.models.detection import _utils as det_utils

In [ ]:
#@title Device 및 속도 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = device.type == "cuda"
AMP_ENABLED = device.type == "cuda"

if AMP_ENABLED:
    # 모든 원본 이미지가 같은 크기이므로, 첫 batch 이후 더 빠른 cuDNN kernel을 사용
    torch.backends.cudnn.benchmark = True

# balanced=True: 속도를 우선하는 448px 입력. mAP가 많이 떨어지면 False로 바꾸세요.
USE_BALANCED_SPEED = False
MODEL_MIN_SIZE = 448 if USE_BALANCED_SPEED else 512
MODEL_MAX_SIZE = 640 if USE_BALANCED_SPEED else 800
NUM_EPOCHS = 100  # 기존 실행에서 최고 mAP가 epoch 19였음
VAL_EVERY = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
NUM_WORKERS = 0  # Windows/Jupyter 안정 기본값; 정상 완주 후 2까지 시험 가능

print(f"device: {device} | input min_size: {MODEL_MIN_SIZE}")

In [ ]:
#@title Image, Annotation 경로 설정
kagglehub.login()
path = kagglehub.competition_download("ai14-level-project")
path = os.path.join(path, "sprint_ai_project1_data")
print(path)

image_dir = glob.glob(os.path.join(path, "train_images", "*.png"))

annotation_dir = glob.glob(
    os.path.join(path, "train_annotations", "**", "*.json"),
    recursive=True
)

print(f"Train 이미지 수 : {len(image_dir)}, Annotation 수 : {len(annotation_dir)}")

In [ ]:
#Cell 4 수정(1:1 mapping 코드 -> 1:n mapping 코드)
#@title Image-Annotation Mapping (stem 기준)

# 이미지 stem -> 이미지 경로
image_stem_to_path = {
    os.path.splitext(os.path.basename(image_path))[0]: image_path
    for image_path in image_dir
}

# 이미지 stem -> 여러 annotation JSON 경로
stem_to_paths = defaultdict(list)

for annotation_path in annotation_dir:
    stem = os.path.splitext(os.path.basename(annotation_path))[0]
    stem_to_paths[stem].append(annotation_path)

# 이미지와 annotation이 모두 존재하는 stem만 사용
valid_stems = [
    stem
    for stem in stem_to_paths
    if stem in image_stem_to_path
]

# 매핑되지 않는 데이터 확인
images_without_annotation = (
    set(image_stem_to_path) - set(stem_to_paths)
)

annotations_without_image = (
    set(stem_to_paths) - set(image_stem_to_path)
)

print(f"Annotation이 없는 Train 이미지 수: {len(images_without_annotation)}")
print(f"Train 이미지가 없는 Annotation stem 수: {len(annotations_without_image)}")
print(f"매핑 가능한 이미지 수: {len(valid_stems)}")

In [ ]:
#@title 이미지 해상도 파악
sizes = set()
for img_path in image_dir[:50]:  # 200장이면 전수 조사해도 됨: image_dir 전체로
    with Image.open(img_path) as im:
        sizes.add(im.size)  # (width, height)

print(f"고유 해상도 종류 수: {len(sizes)}")
print(sizes if len(sizes) < 10 else list(sizes)[:10])

In [ ]:
#@title 이미지 단위 Annotation 병합

def build_image_records(valid_stems, stem_to_paths):
    image_records = {}
    category_id_to_name = {}

    for stem in valid_stems:
        img_info = None
        annotations = []

        for annotation_path in stem_to_paths[stem]:
            with open(annotation_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # 이미지 정보는 같은 stem의 JSON끼리 동일하므로 처음 한 번만 저장
            if img_info is None:
                img_info = data["images"][0]

            # 클래스 정보 저장
            for category in data["categories"]:
                category_id_to_name[category["id"]] = category["name"]

            # 같은 이미지에 속한 annotation들을 하나로 병합
            for ann in data["annotations"]:
                annotations.append({
                    "bbox": ann["bbox"],
                    "category_id": ann["category_id"],
                    "area": ann["area"],
                    "iscrowd": ann.get("iscrowd", 0)
                })

        image_records[stem] = {
            "file_name": img_info["file_name"],
            "width": img_info["width"],
            "height": img_info["height"],
            "annotations": annotations
        }

    return image_records, category_id_to_name


image_records, category_id_to_name = build_image_records(
    valid_stems,
    stem_to_paths
)

print(f"이미지 record 수: {len(image_records)}")
print(f"클래스 수: {len(category_id_to_name)}")

In [ ]:
#@title 클래스(category) 매핑 생성
sorted_ids = sorted(category_id_to_name.keys())

category_to_label = {
    cid: i + 1
    for i, cid in enumerate(sorted_ids)
}

label_to_category = {
    v: k
    for k, v in category_to_label.items()
}

classes = ["background"] + [
    category_id_to_name[cid]
    for cid in sorted_ids
]

num_classes = len(classes)

print(f"클래스 수(background 포함): {num_classes}")

In [ ]:
# #@title 폴더별 partial annotation 병합
def load_merged_annotation(paths):
    img_info = None
    boxes, labels, areas, iscrowd = [], [], [], []

    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)

        if img_info is None:
            img_info = data["images"][0]  # width/height/file_name은 모든 폴더에서 동일하다고 가정

        for ann in data["annotations"]:
            boxes.append(ann["bbox"])       # [x, y, w, h]
            labels.append(ann["category_id"])
            areas.append(ann["area"])
            iscrowd.append(ann.get("iscrowd", 0))

    return img_info, boxes, labels, areas, iscrowd

In [ ]:
import pandas as pd
from collections import Counter, defaultdict

class_instance_count = Counter()      # 클래스별 annotation(알약 객체) 개수
class_image_count = defaultdict(set)  # 클래스별 등장하는 이미지(stem) 집합

for stem, paths in stem_to_paths.items():
    img_info, raw_boxes, raw_labels, areas, iscrowd = load_merged_annotation(paths)
    for cid in raw_labels:
        class_instance_count[cid] += 1
        class_image_count[cid].add(stem)

print(f"Train 데이터에 존재하는 고유 클래스 수: {len(class_instance_count)}\n")

for cid in sorted(class_instance_count.keys()):
    name = category_id_to_name[cid]
    print(f"category_id: {cid:>6} | 이름: {name:<20} | "
          f"객체 수: {class_instance_count[cid]:>4} | "
          f"등장 이미지 수: {len(class_image_count[cid]):>4}")

df = pd.DataFrame([
    {
        "category_id": cid,
        "name": category_id_to_name[cid],
        "instance_count": class_instance_count[cid],
        "image_count": len(class_image_count[cid])
    }
    for cid in sorted(class_instance_count.keys())
])
df = df.sort_values("instance_count", ascending=False).reset_index(drop=True)
df

In [ ]:
#@title 클래스별 이미지 수
# Windows 한글 폰트 설정 (맑은 고딕)
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

colors = plt.cm.nipy_spectral(np.linspace(0, 1, len(df)))

plt.figure(figsize=(30, 20))

plt.barh(
    df["name"],
    df["image_count"],
    color=colors
)

plt.xlabel("이미지 수")
plt.ylabel("클래스")
plt.title("클래스별 이미지 수")

plt.gca().invert_yaxis()  # df가 내림차순이면 큰 값이 위로 오게 함
plt.tight_layout();
plt.show();

In [ ]:
#@title Train 이미지 + Bounding Box 시각화 (랜덤 5개) -- Annotation 조정 전

import json

n_samples = 5
sample_paths = np.random.choice(image_dir, size=n_samples, replace=False)

fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 5))

for ax, img_path in zip(axes, sample_paths):
    img_stem = os.path.splitext(os.path.basename(img_path))[0]

    # 같은 파일명의 annotation 찾기
    ann_path = next(
        (p for p in annotation_dir if os.path.splitext(os.path.basename(p))[0] == img_stem),
        None
    )

    train_image = cv2.imread(img_path)
    train_image = cv2.cvtColor(train_image, cv2.COLOR_BGR2RGB)

    if ann_path is not None:
        with open(ann_path, "r", encoding="utf-8") as f:
            ann = json.load(f)

        boxes = []
        if "annotations" in ann:
            for obj in ann["annotations"]:
                if "bbox" in obj:
                    boxes.append(obj["bbox"])  # [xmin, ymin, xmax, ymax] 가정
        elif "shapes" in ann:
            for obj in ann["shapes"]:
                if "points" in obj:
                    xs = [p[0] for p in obj["points"]]
                    ys = [p[1] for p in obj["points"]]
                    boxes.append([min(xs), min(ys), max(xs), max(ys)])

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(train_image, (x1, y1), (x2, y2), (255, 0, 0), 2)
    else:
        print(f"'{img_stem}'에 해당하는 annotation을 찾지 못했습니다.")

    ax.imshow(train_image)
    ax.set_title(img_stem, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#@title Train 이미지 + Bounding Box 시각화 (XYWH → XYXY 변환 후)

n_samples = min(5, len(image_dir))

sample_paths = np.random.choice(
    image_dir,
    size=n_samples,
    replace=False
)

fig, axes = plt.subplots(
    1,
    n_samples,
    figsize=(5 * n_samples, 5)
)

# 이미지가 1장이어도 반복 가능하게 처리
axes = np.atleast_1d(axes)

for ax, img_path in zip(axes, sample_paths):
    img_stem = os.path.splitext(
        os.path.basename(img_path)
    )[0]

    train_image = cv2.imread(img_path)
    train_image = cv2.cvtColor(
        train_image,
        cv2.COLOR_BGR2RGB
    )

    image_h, image_w = train_image.shape[:2]

    # 여러 annotation JSON을 병합한 record 사용
    record = image_records.get(img_stem)

    if record is None:
        print(
            f"'{img_stem}'에 해당하는 annotation을 "
            "찾지 못했습니다."
        )
        boxes = []
    else:
        boxes = []

        for ann in record["annotations"]:
            # 원본 COCO bbox: [x, y, width, height]
            x, y, bw, bh = ann["bbox"]

            # OpenCV용 XYXY 좌표로 변환
            x1 = int(round(x))
            y1 = int(round(y))
            x2 = int(round(x + bw))
            y2 = int(round(y + bh))

            # 이미지 범위를 벗어나지 않도록 제한
            x1 = max(0, min(x1, image_w - 1))
            y1 = max(0, min(y1, image_h - 1))
            x2 = max(0, min(x2, image_w - 1))
            y2 = max(0, min(y2, image_h - 1))

            if x2 <= x1 or y2 <= y1:
                print(
                    f"잘못된 bbox 제외: "
                    f"{img_stem}, {ann['bbox']}"
                )
                continue

            boxes.append([x1, y1, x2, y2])

            cv2.rectangle(
                train_image,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                2
            )

    ax.imshow(train_image)
    ax.set_title(
        f"{img_stem}\nBoxes: {len(boxes)}",
        fontsize=9
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#@title label_to_category 매핑 자체 검증
expected_labels = set(range(1, num_classes))  # background(0) 제외
mapped_labels = set(label_to_category.keys())

print(f"모델이 출력 가능한 라벨 범위: {expected_labels}")
print(f"label_to_category에 존재하는 라벨: {mapped_labels}")
print(f"매핑 누락된 라벨: {expected_labels - mapped_labels}")
print(f"매핑 결과 category_id 샘플 5개: {list(label_to_category.items())[:5]}")

## STEP 2 : Transform ~ Dataset ~ DataLoader

In [ ]:
#@title Transformer
train_transformer = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.02,
    ),
])

val_test_transformer = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
])

In [ ]:
#@title Pill Dataset 정의
class PillDataset(Dataset):
    def __init__(
        self,
        image_dir,
        image_records,
        stems,
        category_to_label,
        transforms=None
    ):
        self.image_dir = image_dir
        self.image_records = image_records
        self.stems = stems
        self.category_to_label = category_to_label
        self.transforms = transforms

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]

        # 미리 병합해둔 annotation 정보 가져오기
        record = self.image_records[stem]

        with Image.open(os.path.join(self.image_dir, record["file_name"])) as im:
            image = im.convert("RGB")

        w = record["width"]
        h = record["height"]

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in record["annotations"]:
            x, y, bw, bh = ann["bbox"]

            boxes.append([x, y, x + bw, y + bh])
            labels.append(
                self.category_to_label[ann["category_id"]]
            )
            areas.append(ann["area"])
            iscrowd.append(ann["iscrowd"])

        image = tv_tensors.Image(image)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        boxes = boxes.reshape(-1, 4) if boxes.numel() else boxes.reshape(0, 4)
        boxes = tv_tensors.BoundingBoxes(
            boxes,
            format="XYXY",
            canvas_size=(h, w)
        )

        target = {
            "boxes": boxes,
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
            "area": torch.as_tensor(areas, dtype=torch.float32),
            "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64),
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target

In [ ]:
#@title Multilabel Stratified Train / Validation Split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

SEED = 42
VAL_RATIO = 0.2
N_CANDIDATES = 200

# 실행 환경마다 glob 순서가 달라지는 것을 방지
stems = np.asarray(sorted(image_records.keys()), dtype=object)

category_ids = np.asarray(
    sorted(category_to_label.keys())
)

category_to_col = {
    int(category_id): col
    for col, category_id in enumerate(category_ids)
}

# ---------------------------------------------------------
# 이미지별 multi-hot label 행렬 생성
# y[i, j] = i번째 이미지에 j번째 클래스가 있으면 1
# 같은 클래스 객체가 여러 개 있어도 이미지 기준으로는 1
# ---------------------------------------------------------
y = np.zeros(
    (len(stems), len(category_ids)),
    dtype=np.uint8
)

for i, stem in enumerate(stems):
    image_category_ids = {
        int(ann["category_id"])
        for ann in image_records[stem]["annotations"]
    }

    for category_id in image_category_ids:
        y[i, category_to_col[category_id]] = 1


# 클래스별 등장 이미지 수
total_image_counts = y.sum(axis=0)

# ---------------------------------------------------------
# 등장 이미지가 1장뿐인 클래스는 train/val 양쪽에 배치 불가능
# 학습 자체가 불가능해지는 것을 막기 위해 해당 이미지를 train에 고정
# ---------------------------------------------------------
singleton_cols = np.flatnonzero(total_image_counts == 1)

if len(singleton_cols) > 0:
    forced_train_mask = y[:, singleton_cols].any(axis=1)
    forced_train_idx = np.flatnonzero(forced_train_mask)
else:
    forced_train_idx = np.array([], dtype=np.int64)

all_idx = np.arange(len(stems))
eligible_idx = np.setdiff1d(all_idx, forced_train_idx)

n_valid = max(1, int(round(len(stems) * VAL_RATIO)))

if len(eligible_idx) <= n_valid:
    raise ValueError(
        "Singleton 클래스 이미지를 train에 고정한 뒤 "
        "validation 후보 이미지가 부족합니다."
    )


# ---------------------------------------------------------
# 여러 random seed 후보 중
# 1. train 누락 클래스가 가장 적고
# 2. 충분한 표본이 있는 클래스의 val 누락이 가장 적고
# 3. 클래스별 val 비율이 20%에 가까운 분할 선택
# ---------------------------------------------------------
min_evaluable_count = int(np.ceil(1 / VAL_RATIO))  # 20%이면 5장
evaluable_classes = total_image_counts >= min_evaluable_count

best_result = None

for random_state in range(SEED, SEED + N_CANDIDATES):
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=n_valid,
        random_state=random_state,
    )

    relative_train_idx, relative_valid_idx = next(
        splitter.split(
            np.zeros((len(eligible_idx), 1)),
            y[eligible_idx],
        )
    )

    train_idx = np.concatenate([
        forced_train_idx,
        eligible_idx[relative_train_idx],
    ])
    valid_idx = eligible_idx[relative_valid_idx]

    train_counts = y[train_idx].sum(axis=0)
    valid_counts = y[valid_idx].sum(axis=0)

    missing_train_count = int(
        (train_counts == 0).sum()
    )

    missing_valid_count = int(
        (valid_counts[evaluable_classes] == 0).sum()
    )

    if evaluable_classes.any():
        validation_ratio_error = np.abs(
            valid_counts[evaluable_classes]
            / total_image_counts[evaluable_classes]
            - VAL_RATIO
        ).mean()
    else:
        validation_ratio_error = 0.0

    score = (
        missing_train_count,
        missing_valid_count,
        validation_ratio_error,
    )

    if best_result is None or score < best_result[0]:
        best_result = (
            score,
            train_idx,
            valid_idx,
        )


best_score, train_idx, valid_idx = best_result

train_stems = stems[train_idx].tolist()
valid_stems = stems[valid_idx].tolist()


# ---------------------------------------------------------
# 분할 결과 검증
# ---------------------------------------------------------
split_stats = pd.DataFrame({
    "category_id": category_ids.astype(int),
    "class_name": [
        category_id_to_name[int(category_id)]
        for category_id in category_ids
    ],
    "all_images": total_image_counts,
    "train_images": y[train_idx].sum(axis=0),
    "valid_images": y[valid_idx].sum(axis=0),
})

split_stats = split_stats.sort_values(
    ["all_images", "category_id"]
).reset_index(drop=True)

print(f"전체 이미지: {len(stems)}")
print(f"Train 이미지: {len(train_stems)}")
print(f"Validation 이미지: {len(valid_stems)}")
print(f"Train 강제 배치 이미지: {len(forced_train_idx)}")
print(f"선택된 split score: {best_score}")

display(split_stats)


# 데이터 누락 및 중복 검증
assert set(train_stems).isdisjoint(valid_stems)
assert len(train_stems) + len(valid_stems) == len(stems)

missing_train = split_stats[
    split_stats["train_images"] == 0
]

missing_valid = split_stats[
    (split_stats["all_images"] >= min_evaluable_count)
    & (split_stats["valid_images"] == 0)
]

if not missing_train.empty:
    display(missing_train)
    raise RuntimeError(
        "Train에 존재하지 않는 클래스가 있습니다."
    )

if not missing_valid.empty:
    display(missing_valid)
    raise RuntimeError(
        f"이미지가 {min_evaluable_count}장 이상인데 "
        "Validation에 없는 클래스가 있습니다."
    )

# 2~4장뿐인 클래스는 validation에 없을 수 있음
rare_missing_valid = split_stats[
    split_stats["all_images"].between(
        2,
        min_evaluable_count - 1
    )
    & (split_stats["valid_images"] == 0)
]

if not rare_missing_valid.empty:
    print(
        "\n주의: 표본이 2~4장뿐이라 Validation에 포함되지 않은 "
        "클래스입니다. 학습 표본 보존을 우선한 정상적인 결과입니다."
    )
    display(rare_missing_valid)

In [ ]:
#@title Dataset ~ DataLoader
train_dataset = PillDataset(
    image_dir=os.path.join(path, "train_images"),
    image_records=image_records, stems=train_stems,
    category_to_label=category_to_label, transforms=train_transformer,
)
valid_dataset = PillDataset(
    image_dir=os.path.join(path, "train_images"),
    image_records=image_records, stems=valid_stems,
    category_to_label=category_to_label, transforms=val_test_transformer,
)

# lambda를 쓰지 않아 Windows multiprocessing에서도 pickle 오류를 피함
def detection_collate_fn(batch):
    return tuple(zip(*batch))

def make_detection_loader(dataset, batch_size, shuffle):
    kwargs = dict(
        dataset=dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=detection_collate_fn, num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    if NUM_WORKERS > 0:
        kwargs.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**kwargs)

train_loader = make_detection_loader(train_dataset, TRAIN_BATCH_SIZE, shuffle=True)
val_loader = make_detection_loader(valid_dataset, EVAL_BATCH_SIZE, shuffle=False)

## STEP 3 : Modelling

### Model & Optimizer

### mAP method

### Train - Validation Loop

### mAP & Loss Visualization

In [ ]:
#@title RetinaNet ResNet50 FPN v2

model = torchvision.models.detection.retinanet_resnet50_fpn_v2(
    weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT,
    min_size=MODEL_MIN_SIZE,
    max_size=MODEL_MAX_SIZE,
    detections_per_img=20,
)

in_channels = model.backbone.out_channels
num_anchors = (
    model.anchor_generator
    .num_anchors_per_location()[0]
)

# pretrained 91-class head를 현재 데이터 클래스 수에 맞게 교체
model.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes,
    norm_layer=partial(torch.nn.GroupNorm, 32),
)

model = model.to(device)

CHECKPOINT_PATH = (
    "best_model_retinanet_resnet50_fpn_v2.pth"
)

In [ ]:
#@title Define Optimizer & Scheduler
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=3e-3,
    momentum=0.9,
    weight_decay=0.0005,
)

lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4,
)

In [ ]:
#@title mAP 계산 함수 (IoU threshold 0.75~0.95)
iou_thresholds = [0.75, 0.80, 0.85, 0.90, 0.95]

def evaluate_map(model, data_loader, device):
    was_training = model.training
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", iou_thresholds=iou_thresholds)

    # evaluation도 AMP를 사용하면 validation 시간이 크게 줄어듭니다.
    with torch.inference_mode(), torch.cuda.amp.autocast(enabled=AMP_ENABLED):
        for images, targets in tqdm(data_loader, desc="Evaluating"):
            images = [img.to(device, non_blocking=PIN_MEMORY) for img in images]
            preds = model(images)

            preds_cpu = [
                {
                    "boxes": p["boxes"].float().cpu(),
                    "scores": p["scores"].float().cpu(),
                    "labels": p["labels"].cpu(),
                }
                for p in preds
            ]
            targets_cpu = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]

            metric.update(preds_cpu, targets_cpu)

    result = metric.compute()
    if was_training:
        model.train()
    return result

In [ ]:
num_epochs = NUM_EPOCHS
best_map = float("-inf")  # mAP가 0이어도 첫 checkpoint는 저장
train_loss_hist = []
val_map_hist = []
val_map_75_hist = []
val_epoch_hist = []

scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0.0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images = [img.to(device, non_blocking=PIN_MEMORY) for img in images]
        targets = [
            {k: v.to(device, non_blocking=PIN_MEMORY) for k, v in t.items()}
            for t in targets
        ]

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())

        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        total_train_loss += losses.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_loss_hist.append(avg_train_loss)

    print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss:.4f}")

    val_map = evaluate_map(model, val_loader, device)
    current_map = val_map["map"].item()
    current_map_75 = val_map["map_75"].item()
    lr_scheduler.step(current_map)

    val_epoch_hist.append(epoch + 1)
    val_map_hist.append(current_map)
    val_map_75_hist.append(current_map_75)
    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"mAP[.75:.95]: {current_map:.4f} | "
        f"mAP@75: {current_map_75:.4f}"
    )

    if current_map > best_map:
        best_map = current_map
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": lr_scheduler.state_dict(),
            "best_map": best_map,
        }, CHECKPOINT_PATH)
        print(f"  → New best model saved (mAP: {best_map:.4f})")

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

epochs = range(1, len(train_loss_hist) + 1)
best_idx = max(range(len(val_map_hist)), key=val_map_hist.__getitem__)
best_epoch = val_epoch_hist[best_idx]
best_score = val_map_hist[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Loss
sns.lineplot(
    x=list(epochs),
    y=train_loss_hist,
    marker="o",
    color="tab:blue",
    ax=axes[0],
)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_xticks(list(epochs))

# Validation mAP@[0.75:0.95]
sns.lineplot(
    x=val_epoch_hist,
    y=val_map_hist,
    marker="o",
    color="tab:orange",
    ax=axes[1],
)
axes[1].scatter(
    best_epoch,
    best_score,
    color="crimson",
    s=90,
    zorder=3,
    label=f"Best: {best_score:.4f} (Epoch {best_epoch})",
)
axes[1].annotate(
    f"{best_score:.4f}",
    xy=(best_epoch, best_score),
    xytext=(8, 8),
    textcoords="offset points",
    color="crimson",
)
axes[1].set_title("Validation mAP@[0.75:0.95]")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mAP")
axes[1].set_ylim(0, 1)
axes[1].set_xticks(val_epoch_hist)
axes[1].legend()

plt.tight_layout()
plt.show()

## STEP 4 : Test

### Test Class 정의

### Visualization Method & Visualization Loop

### Submission CSV 제작 함수

In [ ]:
#@title Test 이미지 경로 설정 및 랜덤 샘플링
test_image_dir = glob.glob(os.path.join(path, "test_images", "*.png"))
print(f"Test 이미지 수: {len(test_image_dir)}")

random.seed(42)
test_sample_paths = random.sample(test_image_dir, min(20, len(test_image_dir)))
test_sample_stems = [os.path.splitext(os.path.basename(p))[0] for p in test_sample_paths]

In [ ]:
class TestDataset(Dataset):
    def __init__(self, image_dir, stems, transforms):
        self.image_dir = image_dir
        self.stems = stems
        self.transforms = transforms

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]

        image = Image.open(
            os.path.join(self.image_dir, f"{stem}.png")
        ).convert("RGB")

        orig_w, orig_h = image.size
        image = tv_tensors.Image(image)

        if self.transforms:
            image = self.transforms(image)

        return image, stem, orig_w, orig_h


test_dataset = TestDataset(
    image_dir=os.path.join(path, "test_images"),
    stems=test_sample_stems,
    transforms=val_test_transformer,
)

test_loader = make_detection_loader(
    test_dataset, batch_size=4, shuffle=False
)

In [ ]:
#@title Define method : visualize_prediction
def visualize_prediction(image, prediction, classes, score_thresh=0.5):
    """
    image (torch.Tensor): 추론에 사용된 이미지 (C, H, W).
    prediction (dict): boxes, labels, scores 포함.
    classes (list): index -> 약 이름 매핑 리스트 (0번 = background).
    """
    image = image.permute(1, 2, 0).cpu().numpy()

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(image)

    for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
        if score > score_thresh:
            x_min, y_min, x_max, y_max = box.tolist()
            width, height = x_max - x_min, y_max - y_min

            rect = patches.Rectangle(
                (x_min, y_min), width, height,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x_min, max(y_min - 5, 0),
                f"{classes[label.item()]} ({score:.2f})",
                color="white", fontsize=9,
                bbox=dict(facecolor="red", alpha=0.6, pad=1)
            )

    ax.axis("off")
    plt.show()

In [ ]:
#@title Test 이미지 20장 추론 및 시각화
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with torch.inference_mode(), torch.cuda.amp.autocast(enabled=AMP_ENABLED):
    for images, stems, orig_ws, orig_hs in tqdm(test_loader, desc="Test Inference"):
        images = [img.to(device, non_blocking=PIN_MEMORY) for img in images]
        predictions = model(images)

        for img, pred, stem, orig_w, orig_h in zip(images, predictions, stems, orig_ws, orig_hs):
            print(f"파일명: {stem}")
            visualize_prediction(img.cpu(), pred, classes)

In [ ]:
#@title Submission CSV 생성
model_name = "retinanet_resnet50_fpn_v2"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
)
model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

all_test_paths = sorted(glob.glob(os.path.join(path, "test_images", "*.png")))
all_test_stems = [
    os.path.splitext(os.path.basename(p))[0]
    for p in all_test_paths
]

submission_test_dataset = TestDataset(
    image_dir=os.path.join(path, "test_images"),
    stems=all_test_stems,
    transforms=val_test_transformer
)

submission_test_loader = make_detection_loader(
    submission_test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False
)

rows = []
annotation_id = 1
score_thresh = 0.3 # 최적 파라미터

with torch.inference_mode(), torch.cuda.amp.autocast(enabled=AMP_ENABLED):
    for images, stems, orig_ws, orig_hs in tqdm(
        submission_test_loader,
        desc="Submission Inference"
    ):
        images = [img.to(device, non_blocking=PIN_MEMORY) for img in images]
        predictions = model(images)

        for pred, stem, orig_w, orig_h in zip(
            predictions, stems, orig_ws, orig_hs
        ):
            image_id = int(stem)

            boxes = pred["boxes"].cpu()
            labels = pred["labels"].cpu()
            scores = pred["scores"].cpu()

            keep = scores >= score_thresh
            boxes = boxes[keep]
            labels = labels[keep]
            scores = scores[keep]

            for box, label, score in zip(boxes, labels, scores):
                x_min, y_min, x_max, y_max = box.tolist()

                x_min = max(0, min(x_min, orig_w))
                x_max = max(0, min(x_max, orig_w))
                y_min = max(0, min(y_min, orig_h))
                y_max = max(0, min(y_max, orig_h))

                bbox_w = x_max - x_min
                bbox_h = y_max - y_min

                if bbox_w <= 0 or bbox_h <= 0:
                    continue

                rows.append([
                    annotation_id,
                    image_id,
                    label_to_category[label.item()],
                    round(x_min, 2),
                    round(y_min, 2),
                    round(bbox_w, 2),
                    round(bbox_h, 2),
                    round(score.item(), 4),
                ])
                annotation_id += 1

submission_filename = f"submission_{model_name}.csv"

with open(submission_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "annotation_id", "image_id", "category_id",
        "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
    ])
    writer.writerows(rows)

print(f"총 {len(rows)}개 예측 저장 완료 → {submission_filename}")